In [223]:
import confnotebook

In [224]:
from pathlib import Path

source = Path("../examples/test/full/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 10
[1] 126164
[2] 14964427_Енисейская ТГК-13-БРАЗ
[3] 14976087_АвеларСолар Тех-БРАЗ-1
[4] 15120979_Форвард Энерго-БРАЗ-1
[5] 15235008_ОГК-2-БРАЗ-1
[6] 25
[7] 33
[8] 4
[9] 44
[10] 7-1
[11] Акт сверки взаимных расчетов №00000379931 от 30.04.2024
[12] Акт сверки №0000
[13] Акт сверки №MOW00-0087974   от 10.06.2024
[14] Акт сверки №ТРБП-000006 от 10.01.2024
[15] АС ВНИИМ Менделеева Д.И. - БРАЗ на 31.12.24
[16] АС ВНИИМ Менделеева Д.И. - РУ на 31.12.24
[17] АС КРЕЗОЛ-САЗ на 31.08.25
[18] АС Охрана Металлург-САЗ на 31.12.25
[19] АС РУ- ВОСЬМОЙ ВЕТРОПАРК
[20] АС Фрейт Линк-БРАЗ на 30.09.25
[21] АСР СДД 2 кв.2024 (подп. к-а)
[22] Браз-Юнигрин Пауэр
[23] документ 23-ИИА-03-01141 от 31_03_2025
[24] документ 23-ИИА-03-01142 от 31_03_2025
[25] ЕВР-НКАЗ
[26] Неформализованный_первичный_документ_23_ИИА_03_00342_от_31_03
[27] Неформализованный_первичный_документ_23_ИИА_03_02596_от_31_03
[28] Неформализованный_первичный_документ_23_ИИА_03_03623_от_31_03
[29] Неформализованный_первичный_до

In [225]:
IDX_FILE = 1

In [226]:
import base64
import time

import requests

BASE_URL = "http://127.0.0.1:8001"
pdf_path = files[IDX_FILE]

document_b64 = base64.b64encode(pdf_path.read_bytes()).decode()
resp = requests.post(f"{BASE_URL}/send_reconciliation_act", json={"document": document_b64})

if not resp.ok:
    print(resp.text)


resp.raise_for_status()

process_id = resp.json()["process_id"]
print(f"process_id: {process_id}")

while True:
    resp = requests.post(f"{BASE_URL}/process_status", json={"process_id": process_id})
    if resp.status_code == 200:
        data = resp.json()
        print(f"seller: {data['seller']}")
        print(f"buyer:  {data['buyer']}")
        print(f"debit:  {len(data['debit'])} entries")
        print(f"credit: {len(data['credit'])} entries")
        break
    elif resp.status_code == 201:
        print("processing...")
        time.sleep(3)
    else:
        raise RuntimeError(resp.json())

process_id: e7b5f462-39b3-421c-82e1-53eda36380bd
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
processing...
seller: БОГУЧАНСКАЯ ГЭС, АО
buyer:  РУСАЛ САЯНОГОРСКИЙ АЛЮМИНИЕВЫЙ ЗАВОД, АО
debit:  38 entries
credit: 38 entries


In [227]:
comments = """
            По данным АО "РУСАЛ Новокузнецк" на 30.09.2023
            задолженность в пользу АО "РУСАЛ Новокузнецк"
            составляет 13 755 023,24 руб.
            С разногласиями, протокол разногласий прилагается.
            Акт сверки проверен ОУФО ОЦО, ООО "РЦУ".
            Исполнитель: Воробьева Оксана Евгеньевна
            Дата:29.01.2025
            """

fill_resp = requests.post(f"{BASE_URL}/fill_reconciliation_act", json={
    "process_id": process_id,
    "comments": comments,
    "debit": data["debit"],
    "credit": data["credit"],
})
if not fill_resp.ok:
    print(fill_resp.json())
else:
    fill_resp.raise_for_status()

    out_path = f"../examples/output/{files[IDX_FILE].stem}_filled.pdf"
    Path(out_path).write_bytes(base64.b64decode(fill_resp.json()["document"]))
    print(out_path)

../examples/output/126164_filled.pdf
